## DVF: Predict Next Mutation
## Author: Xavier VAN AUSLOOS, 9/02/26
## Notebook for preparing dataset (train/test) and training ML models for predicting next mutation
## RandomForestRegressor
## Dataset: all houses in France 2020 - 2025

In [1]:
import pandas as pd
import numpy as np
import json

from pathlib import Path
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib


In [2]:
# Get project root and ensure directories exist
_project_root = Path.cwd() if (Path.cwd() / "src" / "dvf").exists() else Path.cwd().parent
data_dir = _project_root / "data" / "processed"
models_dir = _project_root / "data" / "models"
results_dir = _project_root / "data" / "results"

for d in [data_dir, models_dir, results_dir]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Data dir: {data_dir}")
print(f"Models dir: {models_dir}")
print(f"Results dir: {results_dir}")


Data dir: /Users/xaviervanausloos/projects/ldi_dvf/data/processed
Models dir: /Users/xaviervanausloos/projects/ldi_dvf/data/models
Results dir: /Users/xaviervanausloos/projects/ldi_dvf/data/results


In [3]:
# Load dataset from data/processed/
DATA_PATH = data_dir / "df_grouped_2020_2025_houses_cleaned.csv"
df = pd.read_csv(DATA_PATH)
print(f"✅ Loaded {len(df)} rows for houses mutations located in Ensues")
print(f"Columns: {list(df.columns)}")
df.head()


FileNotFoundError: [Errno 2] No such file or directory: '/Users/xaviervanausloos/projects/ldi_dvf/data/processed/df_grouped_2020_2025_houses_cleaned.csv'

In [ ]:
# Parse mutations JSON and extract features for next mutation prediction
import ast

def parse_mutations(mutations_str):
    """Parse mutations JSON string (or Python dict string) and return list of (date, value) tuples.
    
    Handles both JSON format (double quotes) and Python dict format (single quotes).
    """
    if pd.isna(mutations_str) or not mutations_str:
        return []
    
    try:
        # Try ast.literal_eval first (handles Python-style single quotes)
        mutations = ast.literal_eval(mutations_str)
    except (ValueError, SyntaxError):
        try:
            # Fallback to json.loads (handles JSON double quotes)
            mutations = json.loads(mutations_str)
        except (json.JSONDecodeError, ValueError):
            return []
    
    parsed = []
    for mut in mutations:
        for date_str, value_str in mut.items():
            # Parse date
            date = pd.to_datetime(date_str, format="mixed", dayfirst=True, errors="coerce")
            # Parse value (handle comma decimal)
            value = pd.to_numeric(value_str.replace(",", "."), errors="coerce")
            if pd.notna(date) and pd.notna(value):
                parsed.append((date, value))
    return sorted(parsed, key=lambda x: x[0])  # Sort by date

# Extract mutation features
df["mutations_parsed"] = df["mutations"].apply(parse_mutations)
df["n_mutations"] = df["mutations_parsed"].apply(len)

# Filter: need at least 2 mutations to predict next one
df_predictable = df[df["n_mutations"] >= 2].copy()
print(f"✅ Houses with >= 2 mutations: {len(df_predictable)} / {len(df)}")

# check results for a specific house
# df_predictable[(df_predictable["Section"] == "AT") & (df_predictable["No plan"].astype(str) == "113")]


In [ ]:
df_predictable.head()

In [ ]:
# Create features and target for next mutation prediction
def extract_features(row):
    """Extract features from house and mutation history."""
    muts = row["mutations_parsed"]
    if len(muts) < 2:
        return None
    
    # Historical features
    dates = [m[0] for m in muts]
    values = [m[1] for m in muts]
    
    # Last mutation (will be used as target)
    last_date = dates[-1]
    last_value = values[-1]
    
    # Previous mutations (for features)
    prev_dates = dates[:-1]
    prev_values = values[:-1]
    
    # Time features
    # days_since_first: total days from first mutation to last mutation
    # Example: 06/08/2021 to 28/04/2023 = 630 days
    days_since_first = (last_date - dates[0]).days if len(dates) > 1 else 0
    avg_days_between = np.mean([(dates[i+1] - dates[i]).days for i in range(len(dates)-1)]) if len(dates) > 1 else 0
    
    # Value features
    avg_value = np.mean(prev_values) if prev_values else 0
    value_trend = (last_value - prev_values[0]) / prev_values[0] if len(prev_values) > 0 and prev_values[0] > 0 else 0
       
    
    return {
        "n_prev_mutations": len(prev_values),
        "days_since_first": days_since_first,
        "avg_days_between": avg_days_between,
        "avg_prev_value": avg_value,
        "value_trend": value_trend,
        "last_value": last_value,
        "target_date": last_date,  # Next mutation date (using last as proxy)
        "target_value": last_value,  # Next mutation value
    }

# Extract features
features_list = []
for idx, row in df_predictable.iterrows():
    feat = extract_features(row)
    if feat:
        feat["idx"] = idx
        features_list.append(feat)

print(f"✅ Extracted features for {len(features_list)} houses")


In [ ]:
# Build df_features from features_list and merge with df_predictable (house columns)
df_features = pd.DataFrame(features_list)

# Merge: use original index so left_on="idx" matches right_index
# Exclude calculated feature columns to avoid overwriting
calculated_feature_cols = ["n_prev_mutations", "days_since_first", "avg_days_between",
                           "avg_prev_value", "value_trend", "last_value", "target_date", "target_value"]
cols_to_merge = [col for col in df_predictable.columns
                 if col not in df_features.columns and col not in calculated_feature_cols]

if cols_to_merge:
    df_predictable_for_merge = df_predictable[cols_to_merge]
    df_features = df_features.merge(df_predictable_for_merge, left_on="idx", right_index=True, how="left")
else:
    df_features = df_features.merge(df_predictable[[]], left_on="idx", right_index=True, how="left")

print(f"Features extracted: {len(df_features)} rows")
df_features.head()

In [ ]:
# Prepare features (X) and targets (y)
# Features: house characteristics + mutation history features
feature_cols = [
    "Code postal",
    "Code commune",
    "Surface reelle bati",
    "Surface terrain",
    "Nombre pieces principales",
    "n_prev_mutations",
    "days_since_first",
    "avg_days_between",
    "avg_prev_value",
    "value_trend",
]

# Targets: next mutation date (days from reference) and value
X = df_features[feature_cols].copy()
X = X.fillna(0)  # Fill NaN

# Target 1: Days until next mutation (relative to last known date)
reference_date = df_features["target_date"].max()
y_days = (df_features["target_date"] - reference_date).dt.days

# Target 2: Next mutation value
y_value = df_features["target_value"]

print(f"Features shape: {X.shape}")
print(f"Targets shape: {y_days.shape}, {y_value.shape}")


In [ ]:
# Split into train/test sets
X_train, X_test, y_days_train, y_days_test, y_value_train, y_value_test = train_test_split(
    X, y_days, y_value, test_size=0.2, random_state=42
)

print(f"Train: {len(X_train)} rows")
print(f"Test: {len(X_test)} rows")

# Save train/test sets
X_train.to_csv(data_dir / "X_train.csv", index=False)
X_test.to_csv(data_dir / "X_test.csv", index=False)
y_days_train.to_csv(data_dir / "y_days_train.csv", index=False)
y_days_test.to_csv(data_dir / "y_days_test.csv", index=False)
y_value_train.to_csv(data_dir / "y_value_train.csv", index=False)
y_value_test.to_csv(data_dir / "y_value_test.csv", index=False)

print(f"\nSaved train/test sets to {data_dir}")


In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model 1: Predict days until next mutation
model_days = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_days.fit(X_train_scaled, y_days_train)

# Train model 2: Predict next mutation value
model_value = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_value.fit(X_train_scaled, y_value_train)

print("Models trained successfully")


In [ ]:
# Save ML models
joblib.dump(model_days, models_dir / "model_predict_days.pkl")
joblib.dump(model_value, models_dir / "model_predict_value.pkl")
joblib.dump(scaler, models_dir / "scaler.pkl")

print(f"Saved models to {models_dir}")


In [ ]:
# Test ML models
# Predictions
y_days_pred = model_days.predict(X_test_scaled)
y_value_pred = model_value.predict(X_test_scaled)

# Metrics for days prediction
mae_days = mean_absolute_error(y_days_test, y_days_pred)
rmse_days = np.sqrt(mean_squared_error(y_days_test, y_days_pred))
r2_days = r2_score(y_days_test, y_days_pred)

# Metrics for value prediction
mae_value = mean_absolute_error(y_value_test, y_value_pred)
rmse_value = np.sqrt(mean_squared_error(y_value_test, y_value_pred))
r2_value = r2_score(y_value_test, y_value_pred)

print("=== Days Prediction Metrics ===")
# explain MAE days meaning? 
# MAE (Mean Absolute Error) for days prediction tells us, on average, how many days off our model's predictions are from the actual number of days until the next mutation.
# For example, an MAE of 100 means our model typically predicts the timing of the next transaction within 100 days of the true value.
# RMSE (Root Mean Squared Error) for days prediction gives a measure of how far the model's predictions typically deviate from the actual values, 
# giving more weight to larger errors. 
# Lower values indicate better fit, but RMSE is sensitive to outliers.
# R² (R-squared, or coefficient of determination) indicates how much of the variance 
# in the actual values is explained by the model. An R² value closer to 1.0 means better explanatory power, while 0 means the model does no better than guessing the mean.
# Explain metrics and print values with contextual examples for mutation prediction

print("=== Days Prediction Metrics ===")
print(
    f"MAE (Mean Absolute Error) for days prediction: {mae_days:.2f} days.\n"
    f"  → On average, the model's prediction for days until next mutation is off by about {mae_days:.0f} days.\n"
    f"    Example: If the true next mutation is in 300 days and MAE={mae_days:.0f}, the prediction is typically within ±{mae_days:.0f} days (e.g., predicts 220-380 days).\n"
)
print(
    f"RMSE (Root Mean Squared Error) for days prediction: {rmse_days:.2f} days.\n"
    f"  → Typical deviation for the timing prediction, giving higher weight to large errors, is about {rmse_days:.0f} days.\n"
    f"    Example: A few large outlier errors will make RMSE higher than MAE.\n"
)
print(
    f"R² (variance explained) for days prediction: {r2_days:.4f}.\n"
    f"  → The model explains about {r2_days*100:.1f}% of the variance in actual days until mutation.\n"
    f"    Example: R²={r2_days:.2f} means predictions are much better than simply predicting the average, if close to 1.0.\n"
)

print("\n=== Value Prediction Metrics ===")
print(
    f"MAE (Mean Absolute Error) for value prediction: {mae_value:.2f} €.\n"
    f"  → On average, the model's next value prediction is off by about {mae_value:,.0f} €.\n"
    f"    Example: If the true next sale is €600,000 and MAE={mae_value:,.0f}, prediction is typically within ±{mae_value:,.0f} € (e.g., predicts €500K–€700K).\n"
)
print(
    f"RMSE (Root Mean Squared Error) for value prediction: {rmse_value:.2f} €.\n"
    f"  → Typical deviation, with higher sensitivity to large errors, is about {rmse_value:,.0f} €.\n"
    f"    Example: Occasional large mistakes increase RMSE much more than MAE.\n"
)
print(
    f"R² (variance explained) for value prediction: {r2_value:.4f}.\n"
    f"  → The model explains about {r2_value*100:.1f}% of the observed variation in transaction values.\n"
    f"    Example: R²={r2_value:.2f} means good prediction capacity if near 1.0, or no gain over the mean if near 0.\n"
)




print(f"MAE: {mae_days:.2f} days")
print(f"RMSE: {rmse_days:.2f} days")
print(f"R²: {r2_days:.4f}")

print("\n=== Value Prediction Metrics ===")
print(f"MAE: {mae_value:.2f} €")
print(f"RMSE: {rmse_value:.2f} €")
print(f"R²: {r2_value:.4f}")


In [ ]:
# Save test results
results = {
    "model_days": {
        "mae": float(mae_days),
        "rmse": float(rmse_days),
        "r2": float(r2_days),
    },
    "model_value": {
        "mae": float(mae_value),
        "rmse": float(rmse_value),
        "r2": float(r2_value),
    },
    "n_train": len(X_train),
    "n_test": len(X_test),
}

import json
with open(results_dir / "test_results.json", "w") as f:
    json.dump(results, f, indent=2)

# Save predictions
predictions_df = pd.DataFrame({
    "y_days_test": y_days_test.values,
    "y_days_pred": y_days_pred,
    "y_value_test": y_value_test.values,
    "y_value_pred": y_value_pred,
})
predictions_df.to_csv(results_dir / "predictions.csv", index=False)

print(f"\nSaved results to {results_dir}")


📅 Prédiction du délai avant la prochaine mutation
MAE (Mean Absolute Error) : 359,9 jours

C’est l’erreur absolue moyenne.

👉 En moyenne, le modèle se trompe d’environ 360 jours sur la date de la prochaine mutation.

Exemple :
si la vraie mutation a lieu dans 300 jours, la prédiction sera en général dans une fourchette d’environ ±360 jours.
Donc quelque part entre 0 et ~660 jours (en pratique).

⚠️ Cela représente une erreur très importante : on est proche d’une année d’écart.

RMSE (Root Mean Squared Error) : 408,8 jours

Même idée que le MAE, mais les grosses erreurs comptent beaucoup plus.

👉 L’écart typique est d’environ 409 jours.

Si le RMSE est supérieur au MAE (ce qui est le cas), cela signifie qu’il existe des erreurs très grandes (outliers) qui dé hookup.

R² (variance expliquée) : -0,9075

Ici c’est le point critique.

Rappel :

1 = modèle parfait

0 = pas mieux que prédire la moyenne

< 0 = pire que prédire la moyenne

👉 -0,91 signifie que le modèle est nettement mauvais.
Si tu prédisais simplement :

“le prochain bien sera vendu dans le délai moyen observé dans l’historique”

… tu obtiendrais de meilleurs résultats.

Donc le modèle n’apprend pas de signal utile pour le temps avant mutation.

💰 Prédiction du montant de la prochaine mutation
MAE : 105 179 €

👉 En moyenne, la prédiction du prix est fausse d’environ 105 k€.

Exemple :
pour une vraie vente à 600 000 €, la prédiction tombe souvent entre environ 495 k€ et 705 k€.

Selon le type de bien, cela peut être :

acceptable sur du très haut de gamme

énorme sur des biens standards.

RMSE : 140 153 €

👉 Les grosses erreurs tirent la métrique vers le haut.
Il y a donc probablement quelques prédictions très éloignées du réel.

R² : -0,268

Encore un indicateur clé.

👉 Le modèle est moins performant qu’une simple moyenne des prix.

Mais :

il est moins catastrophique que pour la prédiction du délai

on sent qu’il y a peut-être un peu de signal, mais mal exploité.

🎯 Conclusion synthétique (version décideur)

Sur les données DVF :

La prédiction du temps avant mutation est actuellement inexploitable.

La prédiction de valeur est faible et doit être fortement améliorée avant usage métier.

Une règle naïve basée sur les moyennes ferait mieux.

💡 En pratique sur DVF

Même avec le bon modèle :

la date de revente reste très difficile à prédire.

Pourquoi ?
Parce que la décision de vendre dépend surtout de :

vie personnelle

héritage

divorce

opportunité financière

déménagement pro

→ informations absentes du dataset.